# Prediction Data Normalization

将原始预测数据（Proby xlsx / Tox21 csv）转换为统一的 parquet 格式，供 MyLabData 应用读取。

## 目录约定（可自由修改）

| 类型 | 路径 |
| :--- | :--- |
| 原始数据 | `RAW_PREDICTION_DIR / {Model} / {BatchCode}/*.xlsx / *.csv` |
| 归一化数据 | `NORMALIZED_PREDICTION_DIR / {Model} / {BatchCode}.parquet` |

## 使用流程

1. 将原始预测文件放入对应目录
2. 运行 Cell 2（导入 + 路径配置）并按需修改路径变量
3. 运行 Cell 3（扫描状态）查看哪些 batch 需要归一化
4. 按需运行 Proby 或 Tox21 的归一化 cell
5. 运行最后的校验 cell 确认结果
6. 回到应用点击 **Refresh** 按钮同步

In [ ]:
import os
import re
from pathlib import Path

import pandas as pd
import numpy as np

# ── 路径配置（按需修改） ──
# 原始预测文件目录：按 model / batch 存放 xlsx 或 csv
RAW_PREDICTION_DIR = Path(r"E:\DryData\Prediction")

# 归一化 parquet 输出目录：按 model 输出 {BatchCode}.parquet
NORMALIZED_PREDICTION_DIR = Path(r"E:\DryData\Prediction")

SUPPORTING_DIR = Path(r"C:\Users\Cenking\Documents\SwissTools\MyLabData\Supporting")

def model_raw_dir(model: str) -> Path:
    return RAW_PREDICTION_DIR / model

def model_norm_dir(model: str) -> Path:
    return NORMALIZED_PREDICTION_DIR / model

# ── 模型定义 ──
ACTIVE_MODELS = ["Proby", "Tox21"]

PROBY_RESULT_COLUMNS = [
    "abs", "emi", "plqy", "e", "log10e", "lifetime",
    "abs_fwhm_cm", "emi_fwhm_cm", "abs_fwhm_nm", "emi_fwhm_nm",
]

# raw xlsx 列名 → 归一化列名
_PROBY_COL_RENAME = {
    "abs fwhm (cm-1)": "abs_fwhm_cm",
    "emi fwhm (cm-1)": "emi_fwhm_cm",
    "abs fwhm (nm)": "abs_fwhm_nm",
    "emi fwhm (nm)": "emi_fwhm_nm",
}

# 原始 Proby xlsx 需要保留的列名（rename 前）
_PROBY_RAW_KEEP = [
    "smiles", "abs", "emi", "plqy", "e", "log10e", "lifetime",
    "abs fwhm (cm-1)", "emi fwhm (cm-1)", "abs fwhm (nm)", "emi fwhm (nm)",
]

TOX21_RESULT_COLUMNS = [
    "NR-AR", "NR-AR-LBD", "NR-AhR", "NR-Aromatase",
    "NR-ER", "NR-ER-LBD", "NR-PPAR-gamma",
    "SR-ARE", "SR-ATAD5", "SR-HSE", "SR-MMP", "SR-p53",
]

# ── 溶剂映射 ──
def load_solvent_map():
    md_path = SUPPORTING_DIR / "Solvent.md"
    mapping = {}
    if md_path.exists():
        for line in md_path.read_text(encoding="utf-8").splitlines():
            m = re.match(r"\|\s*`([^`]+)`\s*\|\s*\*\*([^*]+)\*\*\s*\|", line)
            if m:
                mapping[m.group(1)] = m.group(2)
    return mapping

def _parse_sheet_solvent(sheet_name):
    """从 Proby sheet 名（如 'CS(C)=O (8)'）提取溶剂 SMILES。"""
    m = re.match(r"^(.+?)\s*\(\d+\)$", sheet_name)
    return m.group(1) if m else sheet_name

solvent_map = load_solvent_map()
print(f"已加载 {len(solvent_map)} 个溶剂映射")
print(f"原始数据目录: {RAW_PREDICTION_DIR}")
print(f"归一化输出目录: {NORMALIZED_PREDICTION_DIR}")

In [ ]:
# ── 扫描所有 batch 状态（原始目录 + 归一化目录） ──

print("="*60)
print("  Prediction Data Status")
print("="*60)

for model in ACTIVE_MODELS:
    raw_model_dir = model_raw_dir(model)
    norm_model_dir = model_norm_dir(model)

    print(f"\n── {model} ──")
    print(f"  原始目录: {raw_model_dir}")
    print(f"  归一化目录: {norm_model_dir}")

    raw_batches = []
    if raw_model_dir.exists():
        raw_batches = sorted([
            d.name for d in raw_model_dir.iterdir()
            if d.is_dir() and not d.name.startswith("_")
        ])

    norm_batches = []
    if norm_model_dir.exists():
        norm_batches = sorted([p.stem for p in norm_model_dir.glob("*.parquet")])

    batches = sorted(set(raw_batches) | set(norm_batches))

    if not batches:
        print("  (无 batch 数据)")
        continue

    for bc in batches:
        raw_dir = raw_model_dir / bc
        norm_path = norm_model_dir / f"{bc}.parquet"

        if model == "Proby":
            raw_count = len(list(raw_dir.glob("*.xlsx"))) if raw_dir.exists() else 0
            raw_label = f"{raw_count} xlsx"
        else:
            raw_count = len(list(raw_dir.glob("*.csv"))) if raw_dir.exists() else 0
            raw_label = f"{raw_count} csv"

        if norm_path.exists():
            pq_df = pd.read_parquet(norm_path)
            status = f"✓ 已归一化 ({len(pq_df)} rows)"
        else:
            status = "✗ 未归一化"

        print(f"  {bc}: {raw_label} → {status}")

## Proby 归一化

从 `RAW_PREDICTION_DIR/Proby/{batch_code}/` 读取所有 xlsx 文件，每个 sheet 对应一种溶剂，
合并为 long-table 格式（SMILES + Solvent + 10 指标列），
输出到 `NORMALIZED_PREDICTION_DIR/Proby/{batch_code}.parquet`。

**修改下方 `batch_code` 变量后运行即可。**

In [ ]:
# ━━━ 设置要归一化的 batch ━━━
batch_code = "BatchE002"  # ← 修改此处
cpu_cores = 20              # ← 使用核数；None 表示自动使用 os.cpu_count()
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━

from tqdm.notebook import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
import pyarrow.parquet as pq
import shutil

raw_dir = model_raw_dir("Proby") / batch_code
assert raw_dir.exists(), f"目录不存在: {raw_dir}"

out_dir = model_norm_dir("Proby")
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / f"{batch_code}.parquet"

xlsx_files = sorted(raw_dir.glob("*.xlsx"))
print(f"找到 {len(xlsx_files)} 个 xlsx 文件", flush=True)

if not xlsx_files:
    raise ValueError(f"{raw_dir} 下没有找到 xlsx 文件")

# ── 检查点目录：每个 xlsx 完成后写一个小 parquet 作为 checkpoint ──
checkpoint_root = out_dir / "_checkpoints"
checkpoint_dir = checkpoint_root / batch_code
checkpoint_dir.mkdir(parents=True, exist_ok=True)

done_stems = {p.stem for p in checkpoint_dir.glob("*.parquet")}
todo_files = [f for f in xlsx_files if f.stem not in done_stems]
skipped = len(xlsx_files) - len(todo_files)

if skipped > 0:
    print(f"✓ 已有 {skipped} 个文件的检查点，跳过；剩余 {len(todo_files)} 个待处理", flush=True)
else:
    print(f"无已有检查点，将处理全部 {len(todo_files)} 个文件", flush=True)

max_workers = cpu_cores if cpu_cores is not None else (os.cpu_count() or 1)
max_workers = max(1, min(max_workers, len(todo_files))) if todo_files else 1
print(f"使用并发 worker 数: {max_workers}", flush=True)


def _load_proby_xlsx(xlsx_path_str: str):
    """读取单个 xlsx 文件的所有 sheet，返回 (文件名, DataFrame, 行数)。"""
    path_obj = Path(xlsx_path_str)
    xl = pd.ExcelFile(path_obj, engine="openpyxl")
    local_frames = []
    total_rows = 0

    for sheet_name in xl.sheet_names:
        solvent_smiles = _parse_sheet_solvent(sheet_name)
        df = xl.parse(sheet_name)

        col_lower = {c.lower().strip(): c for c in df.columns}
        keep = {}
        for raw_col in _PROBY_RAW_KEEP:
            actual = col_lower.get(raw_col.lower())
            if actual is not None:
                keep[actual] = raw_col

        if "smiles" not in [k.lower() for k in keep.values()]:
            continue

        df = df[list(keep.keys())].rename(
            columns={v: k for k, v in zip(keep.values(), keep.keys())}
        )
        df = df.rename(columns={"smiles": "SMILES"})
        df = df.rename(columns=_PROBY_COL_RENAME)
        df["Solvent"] = solvent_smiles

        local_frames.append(df)
        total_rows += len(df)

    xl.close()

    if local_frames:
        chunk = pd.concat(local_frames, ignore_index=True)
        for col in PROBY_RESULT_COLUMNS:
            if col in chunk.columns:
                chunk[col] = pd.to_numeric(chunk[col], errors="coerce")
    else:
        chunk = pd.DataFrame()

    return path_obj.name, path_obj.stem, chunk, total_rows, len(local_frames)


# ── 阶段 1: 逐文件处理并写入 checkpoint ──
if todo_files:
    file_bar = tqdm(total=len(todo_files), desc="Proby 文件进度", unit="file", dynamic_ncols=True)
    molecule_bar = tqdm(desc="分子处理进度", unit="mol", dynamic_ncols=True)
    processed_molecules = 0

    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        future_map = {
            ex.submit(_load_proby_xlsx, str(xlsx_path)): xlsx_path
            for xlsx_path in todo_files
        }

        for fut in as_completed(future_map):
            src = future_map[fut]
            try:
                filename, stem, chunk, mol_count, sheet_count = fut.result()

                if not chunk.empty:
                    ckpt_path = checkpoint_dir / f"{stem}.parquet"
                    chunk.to_parquet(ckpt_path, index=False)
                    del chunk

                processed_molecules += mol_count
                file_bar.update(1)
                file_bar.set_postfix(file=filename)
                molecule_bar.update(mol_count)
                molecule_bar.set_postfix(total=processed_molecules)
                tqdm.write(f"完成 {filename}: {mol_count} rows, {sheet_count} sheets")
            except Exception as e:
                file_bar.update(1)
                tqdm.write(f"读取失败 {src.name}: {e}")

    file_bar.close()
    molecule_bar.close()
else:
    print("所有文件已在之前的运行中完成，直接进入合并阶段", flush=True)

# ── 阶段 2: 将该 batch 的 checkpoint parquet 合并为最终文件 ──
print("\n正在合并检查点文件...", flush=True)

ckpt_files = sorted(checkpoint_dir.glob("*.parquet"))
if not ckpt_files:
    raise ValueError("没有检查点文件可合并，无法生成归一化结果")

writer = None
total_rows_written = 0
solvent_counts: dict[str, int] = {}

try:
    for ckpt in tqdm(ckpt_files, desc="合并进度", unit="file"):
        table = pq.read_table(ckpt)
        if writer is None:
            writer = pq.ParquetWriter(str(out_path), table.schema)
        writer.write_table(table)

        n = table.num_rows
        total_rows_written += n

        if "Solvent" in table.column_names:
            sol_arr = table.column("Solvent").to_pylist()
            for s in set(sol_arr):
                solvent_counts[s] = solvent_counts.get(s, 0) + sol_arr.count(s)

        del table
finally:
    if writer is not None:
        writer.close()

if total_rows_written == 0:
    raise ValueError("所有 xlsx 文件读取失败，无法生成归一化结果")

# ── 清理本 batch 检查点目录 ──
shutil.rmtree(checkpoint_dir)
if checkpoint_root.exists() and not any(checkpoint_root.iterdir()):
    checkpoint_root.rmdir()
print("✓ 检查点目录已清理", flush=True)

solvents = sorted(solvent_counts.keys())
print(f"\n✓ 写入: {out_path}", flush=True)
print(f"  总行数: {total_rows_written}", flush=True)
print(f"  溶剂数: {len(solvents)}", flush=True)
for s in solvents:
    label = solvent_map.get(s, s)
    print(f"    {label} ({s}): {solvent_counts[s]} rows", flush=True)

## Tox21 归一化

从 `RAW_PREDICTION_DIR/Tox21/{batch_code}/` 读取所有 csv 文件，保留 SMILES + 12 毒性指标列，
输出到 `NORMALIZED_PREDICTION_DIR/Tox21/{batch_code}.parquet`。

**修改下方 `batch_code` 变量后运行即可。**

In [ ]:
# ━━━ 设置要归一化的 batch ━━━
batch_code = "BatchE002"  # ← 修改此处
cpu_cores = 16              # ← 使用核数；None 表示自动使用 os.cpu_count()
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━

from tqdm.notebook import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

raw_dir = model_raw_dir("Tox21") / batch_code
assert raw_dir.exists(), f"目录不存在: {raw_dir}"

csv_files = sorted(raw_dir.glob("*.csv"))
print(f"找到 {len(csv_files)} 个 csv 文件", flush=True)

if not csv_files:
    raise ValueError(f"{raw_dir} 下没有找到 csv 文件")

max_workers = cpu_cores if cpu_cores is not None else (os.cpu_count() or 1)
max_workers = max(1, min(max_workers, len(csv_files)))
print(f"使用并发 worker 数: {max_workers}", flush=True)


def _load_tox21_csv(csv_path: str):
    path_obj = Path(csv_path)
    df_local = pd.read_csv(path_obj)
    keep_cols = ["SMILES"] + [c for c in TOX21_RESULT_COLUMNS if c in df_local.columns]
    df_local = df_local[keep_cols]
    return path_obj.name, df_local, len(df_local), len(keep_cols) - 1


frames = []
processed_molecules = 0

file_bar = tqdm(total=len(csv_files), desc="Tox21 文件进度", unit="file", dynamic_ncols=True)
molecule_bar = tqdm(desc="分子处理进度", unit="mol", dynamic_ncols=True)

with ThreadPoolExecutor(max_workers=max_workers) as ex:
    future_map = {
        ex.submit(_load_tox21_csv, str(csv_path)): csv_path
        for csv_path in csv_files
    }

    for fut in as_completed(future_map):
        src = future_map[fut]
        try:
            filename, df, mol_count, metric_count = fut.result()
            frames.append(df)
            processed_molecules += mol_count

            file_bar.update(1)
            file_bar.set_postfix(file=filename)
            molecule_bar.update(mol_count)
            molecule_bar.set_postfix(total=processed_molecules)

            tqdm.write(f"完成 {filename}: {mol_count} rows, {metric_count} metrics")
        except Exception as e:
            file_bar.update(1)
            tqdm.write(f"读取失败 {src.name}: {e}")

file_bar.close()
molecule_bar.close()

if not frames:
    raise ValueError("所有 csv 文件读取失败，无法生成归一化结果")

combined = pd.concat(frames, ignore_index=True)
for col in TOX21_RESULT_COLUMNS:
    if col in combined.columns:
        combined[col] = pd.to_numeric(combined[col], errors="coerce")

out_dir = model_norm_dir("Tox21")
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / f"{batch_code}.parquet"
combined.to_parquet(out_path, index=False)

print(f"\n✓ 写入: {out_path}", flush=True)
print(f"  总行数: {len(combined)}", flush=True)
print(f"  列: {list(combined.columns)}", flush=True)

## 校验

读取指定的归一化 parquet，显示基本信息和前几行数据。

In [ ]:
# ━━━ 选择要校验的 model 和 batch ━━━
model = "Proby"       # ← "Proby" 或 "Tox21"
batch_code = "BatchG001"  # ← 修改此处
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

pq_path = model_norm_dir(model) / f"{batch_code}.parquet"
assert pq_path.exists(), f"文件不存在: {pq_path}"

df = pd.read_parquet(pq_path)
print(f"文件: {pq_path}")
print(f"行数: {len(df)}")
print(f"列名: {list(df.columns)}")
print()

if model == "Proby" and "Solvent" in df.columns:
    solvents = sorted(df["Solvent"].unique())
    print(f"溶剂数: {len(solvents)}")
    for s in solvents:
        label = solvent_map.get(s, s)
        print(f"  {label}: {len(df[df['Solvent']==s])} rows")
    print()

print("── 数值统计 ──")
result_cols = PROBY_RESULT_COLUMNS if model == "Proby" else TOX21_RESULT_COLUMNS
cols_in_df = [c for c in result_cols if c in df.columns]
display(df[cols_in_df].describe().round(4))

print("\n── 前 5 行 ──")
display(df.head())